# Episode replay analyzer

This notebook reads the actual Kaggle replay JSON for a selected submission. It summarizes each episode’s outcome, seat, opponent, turn length, decision volume, terminal state, and recorded action patterns. It also saves normalized episode and decision records for later analysis.

Only replay data returned by the authenticated public Kaggle API is used. The notebook does not execute competitor code or infer private information that is absent from the replay.

In [ ]:
from pathlib import Path
import json, os, time
from collections import Counter
from datetime import datetime, timezone
import pandas as pd

COMPETITION = 'pokemon-tcg-ai-battle'
SUBMISSION_ID = None  # None = newest scored submission belonging to this account
MAX_EPISODES = 100
DELAY_SECONDS = 0.25
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
OUT = ROOT / 'data' / 'episode_analysis'
RAW = OUT / 'replays'
RAW.mkdir(parents=True, exist_ok=True)

token = Path.home() / '.kaggle' / 'access_token'
if token.exists(): os.environ.setdefault('KAGGLE_API_TOKEN', token.read_text().strip())
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

def field(obj, *names, default=None):
    for name in names:
        value = getattr(obj, name, None)
        if value not in (None, ''): return value
    return default

if SUBMISSION_ID is None:
    mine = [s for s in api.competition_submissions(COMPETITION) or []
            if field(s, 'public_score', '_public_score') not in (None, '')]
    if not mine: raise RuntimeError('No scored submissions found for this account.')
    SUBMISSION_ID = int(field(mine[0], 'ref', 'id', '_ref'))
print('Submission:', SUBMISSION_ID)

In [ ]:
episodes = list(api.competition_list_episodes(int(SUBMISSION_ID)) or [])[:MAX_EPISODES]
episode_index = []
for episode in episodes:
    eid = field(episode, 'id', 'episode_id')
    agents = list(field(episode, 'agents', default=[]) or [])
    mine = next((a for a in agents if str(field(a, 'submission_id')) == str(SUBMISSION_ID)), None)
    episode_index.append({
        'episode_id': int(eid),
        'agent_index': field(mine, 'index'),
        'agents': [{
            'index': field(a, 'index'), 'submission_id': field(a, 'submission_id'),
            'team_name': field(a, 'team_name', 'name')
        } for a in agents]
    })

def replay_path(eid): return RAW / f'episode-{eid}-replay.json'
downloaded = 0
for item in episode_index:
    path = replay_path(item['episode_id'])
    if path.exists() and path.stat().st_size: continue
    try:
        api.competition_episode_replay(item['episode_id'], path=str(RAW))
        candidates = list(RAW.glob(f'*{item["episode_id"]}*replay.json'))
        if candidates and candidates[0] != path: candidates[0].replace(path)
        downloaded += int(path.exists())
        time.sleep(DELAY_SECONDS)
    except Exception as exc:
        item['download_error'] = str(exc)

(OUT / 'episode_index.json').write_text(json.dumps(episode_index, indent=2))
print('Episodes listed:', len(episode_index), '| newly downloaded:', downloaded)

In [ ]:
def decks_from_replay(replay):
    decks = {}
    for step in replay.get('steps', []):
        for index, agent in enumerate(step):
            action = agent.get('action')
            if isinstance(action, list) and len(action) == 60 and index not in decks:
                decks[index] = action
        if len(decks) >= 2: break
    return decks

def action_type(option):
    return str(option.get('type')) if isinstance(option, dict) else type(option).__name__

episode_rows, decision_rows, failures = [], [], []
for item in episode_index:
    path = replay_path(item['episode_id'])
    if not path.exists():
        failures.append({'episode_id': item['episode_id'], 'error': 'missing replay'})
        continue
    try: replay = json.loads(path.read_text())
    except Exception as exc:
        failures.append({'episode_id': item['episode_id'], 'error': str(exc)})
        continue
    steps = replay.get('steps') or []
    rewards = replay.get('rewards') or []
    mine_index = item.get('agent_index')
    if mine_index is None:
        failures.append({'episode_id': item['episode_id'], 'error': 'submission seat not found'})
        continue
    mine_index = int(mine_index)
    reward = rewards[mine_index] if mine_index < len(rewards) else None
    outcome = 'win' if reward == 1 else 'loss' if reward == -1 else 'draw_or_unknown'
    first_player = None
    for probe_step in steps:
        if mine_index < len(probe_step):
            probe_obs = (probe_step[mine_index] or {}).get('observation') or {}
            probe_current = probe_obs.get('current') or {}
            if probe_current.get('firstPlayer') in (0, 1):
                first_player = int(probe_current['firstPlayer'])
                break
    play_order = ('first' if first_player == mine_index else 'second'
                  if first_player in (0, 1) else 'unknown')
    opponent = next((a for a in item['agents'] if a.get('index') != mine_index), {})
    status_counts, action_counts, errors = Counter(), Counter(), []
    decision_count = 0
    for step_index, step in enumerate(steps):
        if mine_index >= len(step): continue
        agent = step[mine_index] or {}
        status_counts[str(agent.get('status'))] += 1
        if agent.get('error'): errors.append(str(agent.get('error')))
        obs = agent.get('observation') or {}
        select = obs.get('select') if isinstance(obs, dict) else None
        if not isinstance(select, dict): continue
        decision_count += 1
        options = select.get('option') or []
        for option in options: action_counts[action_type(option)] += 1
        next_step = steps[step_index + 1] if step_index + 1 < len(steps) else []
        next_agent = next_step[mine_index] if mine_index < len(next_step) else {}
        resulting_action = next_agent.get('action') if isinstance(next_agent, dict) else None
        decision_rows.append({
            'episode_id': item['episode_id'], 'step': step_index, 'outcome': outcome,
            'turn': (obs.get('current') or {}).get('turn'),
            'context': select.get('context'), 'option_count': len(options),
            'min_count': select.get('minCount'), 'max_count': select.get('maxCount'),
            'recorded_action': resulting_action, 'action_alignment': 'next_step',
        })
    terminal = (steps[-1][mine_index].get('observation', {}).get('current', {})
                if steps and mine_index < len(steps[-1]) else {})
    episode_rows.append({
        'episode_id': item['episode_id'], 'outcome': outcome, 'reward': reward,
        'seat_index': mine_index, 'first_player_index': first_player, 'play_order': play_order,
        'opponent_submission_id': opponent.get('submission_id'),
        'opponent_team': opponent.get('team_name'), 'step_count': len(steps),
        'turns': terminal.get('turn'), 'terminal_result': terminal.get('result'),
        'decision_count': decision_count, 'status_counts': dict(status_counts),
        'option_type_counts': dict(action_counts), 'errors': errors,
        'own_deck': decks_from_replay(replay).get(mine_index),
    })

(OUT / 'episodes.jsonl').write_text(''.join(json.dumps(x) + '\n' for x in episode_rows))
(OUT / 'decisions.jsonl').write_text(''.join(json.dumps(x) + '\n' for x in decision_rows))
(OUT / 'failures.json').write_text(json.dumps(failures, indent=2))
print('Parsed episodes:', len(episode_rows), '| decisions:', len(decision_rows), '| failures:', len(failures))

In [ ]:
episodes_df = pd.DataFrame(episode_rows)
if len(episodes_df):
    print('Outcome summary')
    display(episodes_df.groupby('outcome').agg(games=('episode_id','count'), avg_turns=('turns','mean'), avg_decisions=('decision_count','mean')))
    print('Play-order performance (reward is authoritative)')
    display(episodes_df.groupby('play_order').agg(games=('episode_id','count'), wins=('reward', lambda x: (x == 1).sum()), win_rate=('reward', lambda x: (x == 1).mean())))
    print('Opponent performance')
    display(episodes_df.groupby(['opponent_submission_id','opponent_team'], dropna=False).agg(games=('episode_id','count'), wins=('reward', lambda x: (x == 1).sum()), win_rate=('reward', lambda x: (x == 1).mean())).sort_values('games', ascending=False))
    display(episodes_df[['episode_id','outcome','play_order','seat_index','opponent_team','turns','decision_count','terminal_result','errors']])
else:
    print('No replay records available; inspect failures.json and API access.')

## Reading the results

- Losses concentrated in one seat suggest first-player advantage or a setup policy issue.
- Short games with normal terminal results suggest strategic failure; missing/aborted statuses suggest packaging, crash, or timeout issues.
- Repeated losses against one opponent identify a matchup target, but need enough episodes to separate signal from variance.
- The normalized JSONL files preserve episode-level evidence for later policy or replay-model training.